# Coverage and data quality: Australian land parcels + G-NAF

What you want to know before building on this dataset:

1. **What's covered**: which states, how many parcels, and whether every parcel the source published made it in.
2. **Which columns you can rely on**: how complete each harmonised column is in each state.
3. **How well parcels link to addresses**: key matches, address coverage, and how the links were made.
4. **What's in the population**: status, tenure, strata and roads, so you can filter for your use case.
5. **Data quality checks**: uniqueness, validity, internal consistency, area plausibility and timeliness.

Add the companion dataset [Australian Addresses (G-NAF)](https://www.kaggle.com/datasets/museofnephorion/australian-addresses-gnaf) as a second input to also check every address link against the G-NAF release itself.

Every number is computed from the files, so re-running this notebook on a new version of the dataset gives a fresh
report. Pipeline source: [github.com/nephorion/kaggle-au-cadastre](https://github.com/nephorion/kaggle-au-cadastre).

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.compute as pc
import pyarrow.parquet as pq


def find_data():
    """The dataset folder: wherever Kaggle mounts it, else $CADASTRE_DATA, else cadastre_out here or one up."""
    hits = [p for p in glob.glob("/kaggle/input/**/parquet", recursive=True)
            if glob.glob(f"{p}/*_parcels.parquet")]
    if hits:
        return os.path.dirname(hits[0])
    if os.environ.get("CADASTRE_DATA"):
        return os.environ["CADASTRE_DATA"]
    return next((d for d in ("cadastre_out", "../cadastre_out") if os.path.isdir(d)), "cadastre_out")


DATA = find_data()
GNAF = next((os.path.dirname(p) for p in glob.glob("/kaggle/input/**/gnaf_full.parquet", recursive=True)),
            os.environ.get("GNAF_DATA"))   # the companion G-NAF dataset, an optional second input
FILES = {os.path.basename(f).split("_")[0]: f for f in sorted(glob.glob(f"{DATA}/parquet/*_parcels.parquet"))}
summary = pd.read_csv(f"{DATA}/summary.csv").set_index("state")
pct = "{:.1f}%".format
print("dataset:", DATA)
print("states with files:", ", ".join(FILES))
print("G-NAF:", GNAF or "not attached (the G-NAF link check is skipped)")

## 1. Coverage at a glance

Parcels, columns and file size per state. `src_*` columns are the state's own attributes, so their number differs
between states.

In [ ]:
def file_facts(path):
    pf = pq.ParquetFile(path)
    names = pf.schema_arrow.names
    return {"parcels": pf.metadata.num_rows, "columns": len(names),
            "src_columns": sum(n.startswith("src_") for n in names), "parquet_mb": os.path.getsize(path) / 1e6}


glance = pd.DataFrame({st: file_facts(f) for st, f in FILES.items()}).T
glance.loc["TOTAL"] = [glance.parcels.sum(), np.nan, np.nan, glance.parquet_mb.sum()]
missing_states = [st for st in summary.index if st not in FILES]
print(f"{int(glance.loc['TOTAL', 'parcels']):,} parcels in {len(FILES)} states")
if missing_states:
    print("not included:", ", ".join(f"{st} ({summary.loc[st, 'licence']})" for st in missing_states))
glance.style.format({"parcels": "{:,.0f}", "columns": "{:.0f}", "src_columns": "{:.0f}", "parquet_mb": "{:,.1f}"},
                    na_rep="")

### Download completeness

Each state's service reports how many parcels it holds; this compares that count with the parcels in the dataset.
A gap means some parcels failed to download.

In [ ]:
dl = summary.loc[list(FILES), ["features_expected", "parcels_written"]].rename(
    columns={"features_expected": "reported_by_source", "parcels_written": "in_dataset"})
dl["missing"] = dl.reported_by_source - dl.in_dataset
dl["complete"] = 100 * dl.in_dataset / dl.reported_by_source
dl.style.format({"reported_by_source": "{:,.0f}", "in_dataset": "{:,.0f}", "missing": "{:,.0f}",
                 "complete": "{:.3f}%"})

## 2. Column completeness

Share of parcels with a value, per column and state. This comes from the Parquet files' own statistics, so it
reads no data. Red means a state doesn't publish that information: check here before comparing states on a column.

In [ ]:
KEY_COLUMNS = ["gnaf_legal_parcel_id", "point_lon", "lot", "section", "plan", "parcel_label", "title_ref",
               "status", "tenure", "is_strata", "is_road", "is_multipart", "area_m2", "perimeter_m",
               "stated_area_m2", "locality", "feature_name", "source_feature_id", "source_created",
               "source_last_updated", "wkt"]


def filled(path):
    """Percent non-null for each top-level, non-list column, from row-group statistics."""
    md_ = pq.ParquetFile(path).metadata
    nulls, rows = {}, md_.num_rows
    for g in range(md_.num_row_groups):
        rg = md_.row_group(g)
        for c in range(rg.num_columns):
            col = rg.column(c)
            if "." in col.path_in_schema:                  # list columns: element-level counts, skip
                continue
            s = col.statistics
            n = s.null_count if s is not None and s.has_null_count else np.nan
            nulls[col.path_in_schema] = nulls.get(col.path_in_schema, 0) + n
    return {k: 100 * (1 - v / rows) for k, v in nulls.items()}


completeness = pd.DataFrame({st: filled(f) for st, f in FILES.items()})
(completeness.reindex(KEY_COLUMNS).style.format("{:.0f}", na_rep="")
 .background_gradient(cmap="RdYlGn", vmin=0, vmax=100, axis=None))

## 3. Address linkage

The reason to use this dataset over a raw cadastre is the link to G-NAF addresses. Per state:

- **key matched**: the parcel's key exists in G-NAF's `LEGAL_PARCEL_ID`
- **has address**: at least one G-NAF address is linked to the parcel
- **G-NAF addresses linked**: share of the state's G-NAF addresses (with a parcel id) that landed on some parcel

Parcels without an address are mostly roads, strata common property and unaddressed land.

In [ ]:
COLS = ["source_oid", "gnaf_legal_parcel_id", "gnaf_key_matched", "gnaf_address_count", "gnaf_address_detail_pids",
        "gnaf_link_methods", "geometry_group_size", "point_lon", "point_lat", "status", "tenure", "is_strata",
        "is_road", "is_multipart", "area_m2", "stated_area_m2", "locality", "source_last_updated"]
AU_LON, AU_LAT = (95.0, 169.0), (-55.5, -9.0)           # Australia incl. Cocos, Christmas, Norfolk, Macquarie
MAIN_LON, MAIN_LAT = (112.0, 154.5), (-44.5, -9.0)       # mainland and Tasmania


def profile(state, path):
    """Read one state (no geometry) and compute every metric used below."""
    t = pq.read_table(path, columns=COLS)
    n_pids = pc.fill_null(pc.list_value_length(t["gnaf_address_detail_pids"]), 0).to_numpy()
    methods = pc.value_counts(pc.list_flatten(t["gnaf_link_methods"].combine_chunks())).to_pylist()
    df = t.drop_columns(["gnaf_address_detail_pids", "gnaf_link_methods"]).to_pandas()
    n, cnt = len(df), df.gnaf_address_count
    addressed = cnt[cnt > 0]

    link = {"parcels": n, "key_matched": 100 * df.gnaf_key_matched.mean(), "has_address": 100 * (cnt > 0).mean(),
            "addresses_on_parcels": cnt.sum(), "mean_per_addressed": addressed.mean(),
            "median_per_addressed": addressed.median(), "p99_per_addressed": addressed.quantile(0.99),
            "max_per_parcel": cnt.max(), "locality_filled": 100 * df.locality.notna().mean()}
    link_kind = {"key match + address": ((cnt > 0) & df.gnaf_key_matched).sum(),
                 "address, no key match": ((cnt > 0) & ~df.gnaf_key_matched).sum(),
                 "no address": (cnt == 0).sum()}

    lon, lat = df.point_lon, df.point_lat
    area, stated = df.area_m2, df.stated_area_m2
    both = (area > 0) & (stated > 0)
    ratio = (area[both] / stated[both])
    upd = pd.to_datetime(df.source_last_updated)
    dq = {
        "duplicate source_oid": df.source_oid.duplicated().sum(),
        "address count != id list": (cnt.to_numpy() != n_pids).sum(),
        "key matched but no key": (df.gnaf_key_matched & df.gnaf_legal_parcel_id.isna()).sum(),
        "point missing": lon.isna().sum(),
        "point outside Australia": (~lon.between(*AU_LON) | ~lat.between(*AU_LAT))[lon.notna()].sum(),
        "offshore islands": (~lon.between(*MAIN_LON) | ~lat.between(*MAIN_LAT))[lon.notna()].sum(),
        "area missing or <= 0": (area.isna() | (area <= 0)).sum(),
        "area < 1 m²": (area < 1).sum(),
        "area > 100 km²": (area > 1e8).sum(),
        "stated area compared": both.sum(),
        "stated area within 10%": (ratio.between(0.9, 1.1)).sum(),
        "stacked geometry": (df.geometry_group_size > 1).sum(),
        "100+ addresses": (cnt >= 100).sum(),
        "has last-updated date": upd.notna().sum(),
        "newest update": upd.max(),
    }
    mix = {"strata": 100 * df.is_strata.mean(), "road": 100 * df.is_road.mean(),
           "multipart": 100 * df.is_multipart.mean()}
    return {"n": n, "link": link, "link_kind": link_kind, "dq": dq, "mix": mix,
            "methods": {m["values"]: m["counts"] for m in methods},
            "status": df.status.value_counts(dropna=False), "tenure": df.tenure.value_counts(dropna=False),
            "ratio": np.log10(ratio[(ratio > 0)].to_numpy()), "updated_year": upd.dt.year.value_counts()}


P = {}
for st, f in FILES.items():
    P[st] = profile(st, f)
    print(f"profiled {st}: {P[st]['n']:,} parcels")

In [ ]:
link = pd.DataFrame({st: p["link"] for st, p in P.items()}).T
link["gnaf_addresses_linked"] = 100 * summary.loc[list(P), "address_link_rate"]
link.style.format({"parcels": "{:,.0f}", "key_matched": pct, "has_address": pct, "addresses_on_parcels": "{:,.0f}",
                   "mean_per_addressed": "{:.2f}", "median_per_addressed": "{:.0f}", "p99_per_addressed": "{:.0f}",
                   "max_per_parcel": "{:,.0f}", "locality_filled": pct, "gnaf_addresses_linked": pct})

In [ ]:
kind = pd.DataFrame({st: p["link_kind"] for st, p in P.items()}).T
share = 100 * kind.div(kind.sum(axis=1), axis=0)
ax = share.plot.barh(stacked=True, figsize=(9, 3.5), color=["#2a9d8f", "#e9c46a", "#d0d0d0"], width=0.75)
ax.set_xlabel("% of parcels")
ax.set_title("How parcels link to G-NAF addresses")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=3, frameon=False)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### How links were made

`gnaf_link_methods` records how each parcel got its addresses: `legal_parcel_id` is the key join, `spatial_within`
the point-in-polygon join (WA, whose public layer has no lot/plan attributes). Counts are parcels using each method.

In [ ]:
methods = pd.DataFrame({st: p["methods"] for st, p in P.items()}).T.fillna(0).astype(int)
methods.style.format("{:,}")

### Links checked against G-NAF

With [Australian Addresses (G-NAF)](https://www.kaggle.com/datasets/museofnephorion/australian-addresses-gnaf) attached, every linked `gnaf_address_detail_pid` is looked up in
the release:

- **found in G-NAF**: share of linked ids that exist in G-NAF. It should be 100%; anything less means the parcels
  were linked to a different G-NAF release than the one attached.
- **on a parcel**: share of the state's G-NAF addresses that are linked to at least one parcel, counted over all
  addresses, including those G-NAF gives no parcel id.
- **on 2+ parcels**: addresses linked to more than one parcel, typically one address on stacked strata lots.

In [ ]:
gnaf_check = None
if GNAF:
    g = pq.read_table(f"{GNAF}/gnaf_full.parquet", columns=["address_detail_pid", "state_abbreviation"])
    all_pids = g.column("address_detail_pid").combine_chunks()
    rows = {}
    for st, path in FILES.items():
        ids = pc.list_flatten(pq.read_table(path, columns=["gnaf_address_detail_pids"]).column(0)).combine_chunks()
        counts = pc.value_counts(ids)
        unique_ids = counts.field("values")
        state_pids = pc.filter(all_pids, pc.equal(g.column("state_abbreviation").combine_chunks(), st))
        rows[st] = {
            "linked ids": len(ids),
            "found in G-NAF": 100 * pc.sum(pc.is_in(ids, value_set=all_pids)).as_py() / max(len(ids), 1),
            "G-NAF addresses": len(state_pids),
            "on a parcel": 100 * pc.sum(pc.is_in(state_pids, value_set=unique_ids)).as_py() / max(len(state_pids), 1),
            "on 2+ parcels": pc.sum(pc.greater(counts.field("counts"), 1)).as_py(),
        }
    gnaf_check = pd.DataFrame(rows).T
    missing = gnaf_check["found in G-NAF"] < 100
    print("PASS: every linked id is in G-NAF" if not missing.any()
          else "FAIL: ids not in G-NAF in " + ", ".join(gnaf_check.index[missing]))
    display(gnaf_check.style.format({"linked ids": "{:,.0f}", "found in G-NAF": "{:.2f}%",
                                     "G-NAF addresses": "{:,.0f}", "on a parcel": "{:.1f}%",
                                     "on 2+ parcels": "{:,.0f}"}))
else:
    print("Add the Australian Addresses (G-NAF) dataset as a second input to run this check.")

## 4. What's in the population

Before analysing, decide which parcels you want. Strata lots, roads and retired or proposed parcels behave very
differently from ordinary freehold lots.

In [ ]:
mix = pd.DataFrame({st: p["mix"] for st, p in P.items()}).T
mix.style.format(pct, na_rep="not published")

In [ ]:
def shares(key):
    t = pd.DataFrame({st: p[key] for st, p in P.items()}).fillna(0)
    t.index = t.index.fillna("(null)") if hasattr(t.index, "fillna") else t.index
    return 100 * t / t.sum()


print("status, % of parcels")
display(shares("status").style.format("{:.1f}"))
print("tenure, % of parcels")
display(shares("tenure").style.format("{:.1f}"))

## 5. Data quality checks

### Scorecard

The first four checks should always be zero; anything else is a defect. "Outside Australia" uses Australia's full
extent, including Lord Howe, Norfolk, Christmas, Cocos (Keeling) and Macquarie Islands.

The second table holds counts to be aware of, not errors: tiny or huge parcels, stacked geometries (strata lots
sharing one footprint) and parcels on offshore islands are all real.

In [ ]:
dq = pd.DataFrame({st: p["dq"] for st, p in P.items()}).T
MUST_BE_ZERO = ["duplicate source_oid", "address count != id list", "key matched but no key",
                "point outside Australia"]
score = pd.DataFrame({
    "check": MUST_BE_ZERO,
    "failures": [int(dq[c].sum()) for c in MUST_BE_ZERO],
})
score["result"] = np.where(score.failures == 0, "PASS", "FAIL")
display(score.style.map(lambda v: "color: #2a9d8f; font-weight: bold" if v == "PASS"
                        else "color: #e76f51; font-weight: bold" if v == "FAIL" else "", subset=["result"]))

n = pd.Series({st: p["n"] for st, p in P.items()})
info = pd.DataFrame({
    "point missing": 100 * dq["point missing"] / n,
    "area missing or <= 0": 100 * dq["area missing or <= 0"] / n,
    "area < 1 m²": 100 * dq["area < 1 m²"] / n,
    "area > 100 km²": dq["area > 100 km²"],
    "stacked geometry": 100 * dq["stacked geometry"] / n,
    "100+ addresses": dq["100+ addresses"],
    "offshore islands": dq["offshore islands"],
})
info.style.format({"point missing": "{:.2f}%", "area missing or <= 0": "{:.3f}%", "area < 1 m²": "{:.2f}%",
                   "area > 100 km²": "{:,.0f}", "stacked geometry": "{:.1f}%", "100+ addresses": "{:,.0f}",
                   "offshore islands": "{:,.0f}"})

### Area plausibility

Where a state publishes the area recorded on the title (`stated_area_m2`), it can be checked against the area
computed from the polygon (`area_m2`). A narrow spike at 1.0 means the geometry is faithful; a wide spread means
the polygons are approximate or the stated areas are stale.

In [ ]:
area_check = pd.DataFrame({
    "compared": dq["stated area compared"],
    "within 10%": 100 * dq["stated area within 10%"] / dq["stated area compared"].replace(0, np.nan),
})
display(area_check.style.format({"compared": "{:,.0f}", "within 10%": "{:.1f}%"}, na_rep="not published"))

with_area = [st for st, p in P.items() if len(p["ratio"])]
if with_area:
    fig, ax = plt.subplots(figsize=(9, 3.5))
    bins = np.linspace(np.log10(0.8), np.log10(1.25), 91)
    for st in with_area:
        ax.hist(P[st]["ratio"], bins=bins, histtype="step", density=True, linewidth=1.5, label=st)
    ax.set_xticks(np.log10([0.8, 0.9, 1, 1.1, 1.25]), ["0.8", "0.9", "1", "1.1", "1.25"])
    ax.set_xlabel("computed area / stated area (log scale)")
    ax.set_yticks([])
    ax.set_title("Polygon area vs area on the title")
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()

### Timeliness

When each state last edited its parcels (`source_last_updated`, where the state publishes it). Most parcels are
old and unchanged; what matters is that recent edits are present, which shows the snapshot is current.

Some states stamp every parcel with the date the whole layer was refreshed. When most parcels share one year, the
date says when the layer was reloaded, not when each parcel changed, so don't use it to find recent edits there.

In [ ]:
years = pd.DataFrame({st: p["updated_year"] for st, p in P.items()}).sort_index().dropna(axis=1, how="all")
top_year_share = 100 * years.max() / years.sum()
fresh = pd.DataFrame({
    "has date": 100 * dq["has last-updated date"] / n,
    "newest edit": dq["newest update"],
    "parcels in busiest year": top_year_share,
})
fresh["date means"] = np.where(fresh["has date"] == 0, "not published",
                               np.where(fresh["parcels in busiest year"] > 90, "layer refresh", "parcel edit"))
display(fresh.style.format({"has date": "{:.0f}%", "parcels in busiest year": "{:.0f}%",
                            "newest edit": lambda d: "" if pd.isna(d) else f"{d:%Y-%m-%d}"}, na_rep=""))

edits = years[[st for st in years.columns if top_year_share[st] <= 90]]
edits = 100 * edits / edits.sum()
edits = edits.loc[edits.index >= edits.index.max() - 15]
if not edits.empty:
    ax = edits.plot(figsize=(9, 3.5), marker="o", markersize=3)
    ax.set_ylabel("% of the state's parcels")
    ax.set_xlabel("year last edited")
    ax.set_title("Parcel edits by year (last 15 years; states with per-parcel dates)")
    ax.legend(frameon=False, ncol=len(edits.columns))
    plt.tight_layout()
    plt.show()

## Summary

In [ ]:
total = int(n.sum())
linked = sum(p["link_kind"]["key match + address"] + p["link_kind"]["address, no key match"] for p in P.values())
print(f"- {total:,} parcels in {len(P)} states; not included: {', '.join(missing_states) or 'none'}")
gap = dl.missing[dl.missing != 0]
print(f"- download completeness: {int(dl.in_dataset.sum()):,} of {int(dl.reported_by_source.sum()):,} parcels the "
      "sources reported; " + (", ".join(f"{st} {int(m):,} missing" for st, m in gap.items()) or "none missing"))
print(f"- {100 * linked / total:.1f}% of parcels have at least one G-NAF address")
if gnaf_check is not None:
    on = gnaf_check["on a parcel"]
    print(f"- linked address ids found in G-NAF: {gnaf_check['found in G-NAF'].min():.2f}% in the lowest state; "
          f"G-NAF addresses on a parcel: {on.min():.1f}% ({on.idxmin()}) to {on.max():.1f}% ({on.idxmax()})")
print(f"- consistency checks: {(score.result == 'PASS').sum()} of {len(score)} pass")
key = completeness.reindex(KEY_COLUMNS)
everywhere = key[(key >= 90).all(axis=1)].index.tolist()
print(f"- columns at least 90% filled in every state: {', '.join(everywhere) or 'none'}")
print("- columns at least 90% filled, per state: "
      + ", ".join(f"{st} {int((key[st] >= 90).sum())}/{len(key)}" for st in key.columns))